# Hebrew Acronym Disambiguation: Evaluation Protocol

This computational appendix evaluates Hebrew acronym disambiguation under two task formulations: **free generation**, without candidate expansions or reference answers, and **candidate selection**, with a fixed candidate inventory and recorded candidate order. Each language model is evaluated on the same items in both tasks.

The technical pilot uses the first **10 of 289 development items** in `data/splits/dev_items.csv`. The test set contains **395 items**: 267 from Knesset text, 41 from Wikipedia, and 87 AI-authored items. It is therefore not a natural-text-only benchmark. Source provenance is documented in `data/README.md` and `data/mined/DATASET_CARD.md` in the source archive. The separate 62-item encoder development cohort is not the pilot cohort used here.

The protocol separates environment preparation, technical pilot inspection, and test evaluation. Software checks establish implementation properties; they do not establish live service availability, successful collection, or model performance. Results are supported by the identified run artifacts produced during execution. The operational reference is `docs/reproducible_evaluation.md` in the source archive.


## 1. Source and Environment Reproducibility

`CODE_REVISION` identifies an exact 40-character Git commit. In archive mode, upload `evaluation-release.tar` through the Colab Files panel and set `ARCHIVE_PATH`; `ARCHIVE_SHA256` must match the independently supplied release checksum. The installation checks both the archive checksum and its embedded commit before installing the package. Archive mode does not require publishing the commit. The alternative Git mode requires that the specified commit be accessible remotely.

The installation verifies the project's pinned dependencies and reports conflicts among other preinstalled Colab packages separately. Restart the runtime and repeat setup after changing the source revision or when Colab requests a restart for dependencies already loaded in memory. Subsequent cells use the installed package.


In [ ]:
import hashlib
import importlib.metadata
import json
import os
from pathlib import Path
import re
import shutil
import subprocess
import sys
import tarfile
import tempfile
import time
import urllib.request

CODE_REVISION = ""  # Exact 40-character source commit; branch names are not valid identities.
SOURCE_MODE = "archive"  # "archive" or "git"
ARCHIVE_PATH = Path("/content/evaluation-release.tar")
ARCHIVE_SHA256 = ""  # Independently supplied release checksum.
REPOSITORY_URL = "https://github.com/BenCarmel123/hebrew-acronym-disambiguation.git"
if not re.fullmatch(r"[0-9a-f]{40}", CODE_REVISION):
    raise ValueError("Set the reviewed full commit SHA before installation")
REPO_DIR = Path("/content/hebrew-eval-" + CODE_REVISION)
REPO_DIR.mkdir(parents=True, exist_ok=True)
if SOURCE_MODE == "archive":
    if not re.fullmatch(r"[0-9a-f]{64}", ARCHIVE_SHA256):
        raise ValueError("Set the release archive SHA-256")
    archive_bytes = ARCHIVE_PATH.read_bytes()
    if hashlib.sha256(archive_bytes).hexdigest() != ARCHIVE_SHA256:
        raise ValueError("Uploaded archive differs from the reviewed release")
    archive_revision = subprocess.check_output(
        ["git", "get-tar-commit-id"], input=archive_bytes).decode().strip()
    if archive_revision != CODE_REVISION:
        raise ValueError("Archive commit differs from CODE_REVISION")
    with tarfile.open(ARCHIVE_PATH, "r:") as archive:
        archive.extractall(REPO_DIR, filter="data")
elif SOURCE_MODE == "git":
    subprocess.run(["git", "init", str(REPO_DIR)], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1",
                    REPOSITORY_URL, CODE_REVISION], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", "FETCH_HEAD"], check=True)
    actual = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
    if actual != CODE_REVISION:
        raise ValueError("Fetched source differs from CODE_REVISION")
else:
    raise ValueError("Choose archive or git")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", str(REPO_DIR)], check=True)
# Regular installation is immediately importable in this running notebook kernel.
# Check this project's pins; Colab can contain unrelated preinstalled conflicts.
for line in (REPO_DIR / "requirements.txt").read_text().splitlines():
    requirement = line.strip()
    if requirement and not requirement.startswith("#"):
        package_name, required_version = requirement.split("==", 1)
        if importlib.metadata.version(package_name) != required_version:
            raise RuntimeError(f"Required package version mismatch: {package_name}")
pip_report = subprocess.run([sys.executable, "-m", "pip", "check"],
                            capture_output=True, text=True)
if pip_report.returncode:
    print("Colab environment dependency report (inspect separately from project pins):")
    print(pip_report.stdout or pip_report.stderr)
print("Installed source commit:", CODE_REVISION)

## 2. Persistent Storage, Credentials, and Expenditure

`RUN_NAME` identifies the persistent Drive directory. Retain this name and the same settings when resuming after a disconnection or selecting additional systems. A changed code revision or configuration requires a new identified session; preserve previous directories and carry their expenditure into the new session exactly once.

Set `PRIOR_SPEND_ILS` to earlier API expenditure outside this session, including failed requests and conservative allowances for uncertain charges. Its initial value, `None`, requires explicit reconciliation before execution. The combined expenditure ceiling is **100 ILS** across systems, pilots, test requests, and retries. The conversion allowance of 4 ILS/USD includes a fee and tax margin; it is not a quoted exchange rate. Changing system selection does not reset expenditure.

`SELECTED_SYSTEMS` determines which of the four systems are attempted. Store `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, and `GEMINI_API_KEY` in **Colab Secrets** and enable access for this notebook. Credentials must not appear in notebook cells or outputs. Missing credentials are reported per system and do not prevent other systems from proceeding. Use one writer per run directory; mounted Drive storage does not guarantee durability during a storage failure. Paid runtime purchases and automatic credit replenishment are outside this protocol.


In [ ]:
from google.colab import drive, userdata
import importlib.metadata
import platform
import torch
from hebrew_acronyms.staged_evaluation import (
    SYSTEM_NAMES, prepare_session, run_system_pilot, review_system_pilot,
    run_system_full, session_summary,
)
from hebrew_acronyms.test_evaluation import summarize_evaluation
from hebrew_acronyms.models.qwen.eval import inspect_ollama
from hebrew_acronyms.models.openai.eval import inspect_openai_model
from hebrew_acronyms.models.anthropic.eval import inspect_anthropic_model
from hebrew_acronyms.models.common.http import request_json

RUN_NAME = "evaluation-staged-20261009-01"
SELECTED_SYSTEMS = ["qwen", "gemini", "openai", "anthropic"]
PRIOR_SPEND_ILS = None  # Required: reconciled prior expenditure and uncertain-cost allowances.
ILS_PER_USD_ALLOWANCE = 4.0  # Conversion allowance including fees and taxes; not a quoted exchange rate.
if not re.fullmatch(r"[A-Za-z0-9_-]+", RUN_NAME):
    raise ValueError("Use a simple stable run name")
if not SELECTED_SYSTEMS or len(set(SELECTED_SYSTEMS)) != len(SELECTED_SYSTEMS) or not set(SELECTED_SYSTEMS) <= set(SYSTEM_NAMES):
    raise ValueError("Select one or more of the four approved systems, without duplicates")
if PRIOR_SPEND_ILS is None:
    raise ValueError("Account for earlier attempts in PRIOR_SPEND_ILS before proceeding")
drive.mount("/content/drive")
OUTPUT_ROOT = Path("/content/drive/MyDrive/NLP/evaluation-runs") / RUN_NAME
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
AVAILABILITY = {name: {"status": "not_selected"} for name in SYSTEM_NAMES}
SECRET_NAMES = {"gemini": "GEMINI_API_KEY", "openai": "OPENAI_API_KEY", "anthropic": "ANTHROPIC_API_KEY"}
for name in SELECTED_SYSTEMS:
    if name not in SECRET_NAMES:
        continue
    secret_name = SECRET_NAMES[name]
    os.environ.pop(secret_name, None)  # Remove stale credentials before reading this notebook's Secrets.
    try:
        secret_value = userdata.get(secret_name)
        if not secret_value:
            raise ValueError("empty")
        os.environ[secret_name] = secret_value
        AVAILABILITY[name] = {"status": "credentials_loaded"}
        del secret_value
    except Exception:
        AVAILABILITY[name] = {"status": "unavailable", "reason": "missing_secret"}
ENVIRONMENT = {
    "code_revision": CODE_REVISION, "python": platform.python_version(),
    "packages": {name: importlib.metadata.version(name) for name in
                 ("torch", "transformers", "requests", "numpy")},
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "archive_sha256": ARCHIVE_SHA256 if SOURCE_MODE == "archive" else None,
}
with (OUTPUT_ROOT / f"environment-{time.time_ns()}.json").open("x") as stream:
    json.dump(ENVIRONMENT, stream, indent=2)
print(ENVIRONMENT)
print("Persistent outputs:", OUTPUT_ROOT)
print(AVAILABILITY)

## 3. Local Model Configuration

Selecting `qwen` installs Ollama and downloads **Qwen2.5 7B** (`qwen2.5:7b`). Use a free GPU when available. Before the pilot, inspect the resolved model digest, server version, template, quantization, stop sequences, and generation settings. These properties identify the local model used in the run; the tag alone is insufficient.

Generation uses temperature 0, seed 42, and a 512-token output limit. Hardware and implementation differences may still affect reproducibility. A setup failure is recorded without substituting another model, and other available systems may proceed.


In [ ]:
QWEN_MODEL = "qwen2.5:7b"
QWEN_OPTIONS = {"temperature": 0, "seed": 42, "num_predict": 512}
OLLAMA_URL = "http://localhost:11434"
QWEN_IDENTITY = None
QWEN_STABLE_FIELDS = ("model", "digest", "server_version", "options",
                      "template", "model_parameters", "details", "model_info")
QWEN_FROZEN_IDENTITY = None
import requests
if "qwen" in SELECTED_SYSTEMS:
    try:
        if shutil.which("ollama") is None:
            subprocess.run(["apt-get", "-qq", "update"], check=True)
            subprocess.run(["apt-get", "-qq", "install", "-y", "zstd"], check=True)
            installer = Path("/content/ollama-install.sh")
            urllib.request.urlretrieve("https://ollama.com/install.sh", installer)
            subprocess.run(["sh", str(installer)], check=True)
        import requests
        try:
            requests.get(OLLAMA_URL + "/api/version", timeout=5).raise_for_status()
        except requests.RequestException:
            ollama_log = (OUTPUT_ROOT / f"ollama-{time.time_ns()}.log").open("w")
            ollama_process = subprocess.Popen(["ollama", "serve"], stdout=ollama_log, stderr=ollama_log)
            for _ in range(30):
                time.sleep(1)
                try:
                    requests.get(OLLAMA_URL + "/api/version", timeout=2).raise_for_status()
                    break
                except requests.RequestException:
                    pass
            else:
                raise RuntimeError("Ollama did not start; inspect its saved log")
        subprocess.run(["ollama", "pull", QWEN_MODEL], check=True)
        QWEN_IDENTITY = inspect_ollama(QWEN_MODEL, options=QWEN_OPTIONS)
        # Download timestamps are session evidence, separate from stable model identity.
        with (OUTPUT_ROOT / f"qwen-inspection-{time.time_ns()}.json").open("x") as stream:
            json.dump(QWEN_IDENTITY, stream, ensure_ascii=False, indent=2)
        QWEN_FROZEN_IDENTITY = {key: QWEN_IDENTITY[key] for key in QWEN_STABLE_FIELDS}
        AVAILABILITY["qwen"] = {"status": "available"}
        print(json.dumps(QWEN_FROZEN_IDENTITY, ensure_ascii=False, indent=2))
    except Exception:
        AVAILABILITY["qwen"] = {"status": "unavailable", "reason": "local_model_setup_failed"}
print(AVAILABILITY["qwen"])

## 4. API Model Availability and Cost Assumptions

The API systems are **Gemini** (`gemini-3.8-flash`), **GPT-4.1 mini** (`gpt-4.1-mini-2025-04-14`), and **Claude Haiku 5.5** (`claude-haiku-5-5`). Availability lookups do not generate text and do not establish sufficient credit or quota. The report distinguishes unavailable and unselected systems; only available selected systems enter collection.

Gemini uses low thinking, and Claude uses adaptive thinking with low effort. Output limits include provider-billed thinking where applicable. Exact settings are recorded with the run, and returned model identities are inspected in the pilot.

The rates below are expressed in USD per million tokens. Verify them against the official [Gemini pricing](https://ai.google.dev/gemini-api/docs/pricing), [GPT-4.1 mini specification](https://developers.openai.com/api/docs/models/gpt-4.1-mini), and [Claude Haiku 5.5 specification](https://platform.claude.com/docs/en/models/haiku-5-5/overview) before collection. Cached input is conservatively charged at the full input rate. Token-based expenditure is an estimate from reported usage, not a settled provider invoice.


In [ ]:
SYSTEMS = [
    {"name": "qwen", "provider": "qwen", "model": QWEN_MODEL,
     "settings": {"expected_digest": (QWEN_IDENTITY or {}).get("digest"), "base_url": OLLAMA_URL,
                  "timeout": 120, "options": QWEN_OPTIONS}},
    {"name": "gemini", "provider": "gemini", "model": "gemini-3.8-flash",
     "settings": {"timeout": 120, "generation_config": {
         "maxOutputTokens": 1024,
         "thinkingConfig": {"thinkingLevel": "low"}}}},
    {"name": "openai", "provider": "openai", "model": "gpt-4.1-mini-2025-04-14",
     "settings": {"timeout": 120, "max_output_tokens": 512, "temperature": 0}},
    {"name": "anthropic", "provider": "anthropic", "model": "claude-haiku-5-5",
     "settings": {"timeout": 120, "max_output_tokens": 1024, "effort": "low"}},
]
RATES = {"qwen": {"input": 0.0, "output": 0.0},
         "gemini": {"input": 0.75, "output": 3.75},
         "openai": {"input": 0.40, "output": 1.60},
         "anthropic": {"input": 0.10, "output": 0.50}}
for system in SYSTEMS:
    name = system["name"]
    if name not in SELECTED_SYSTEMS or name == "qwen" or AVAILABILITY[name]["status"] == "unavailable":
        continue
    if name == "openai":
        info = inspect_openai_model(model=system["model"])
    elif name == "anthropic":
        info = inspect_anthropic_model(model=system["model"])
    else:
        payload, info = request_json(
            requests.get, "https://generativelanguage.googleapis.com/v1beta/models/" + system["model"],
            provider="Gemini", secret=os.environ["GEMINI_API_KEY"], env_name="GEMINI_API_KEY",
            headers={"x-goog-api-key": os.environ["GEMINI_API_KEY"]}, timeout=30)
        if payload is not None and payload.get("name") == "models/" + system["model"] and "generateContent" in (payload.get("supportedGenerationMethods") or []):
            info["status"] = "available"
    # Save only allowlisted diagnostics, never upstream body/message or credentials.
    AVAILABILITY[name] = {key: info.get(key) for key in
        ("status", "http_status", "error_category", "provider_blocked", "retry_after")}
def available_selected_systems():
    # Recheck current selection and availability before collection.
    if not SELECTED_SYSTEMS or len(set(SELECTED_SYSTEMS)) != len(SELECTED_SYSTEMS) or not set(SELECTED_SYSTEMS) <= set(SYSTEM_NAMES):
        raise ValueError("Select approved systems without duplicates")
    return [s for s in SYSTEMS if s["name"] in SELECTED_SYSTEMS
            and AVAILABILITY.get(s["name"], {}).get("status") == "available"]
with (OUTPUT_ROOT / f"availability-{time.time_ns()}.json").open("x") as stream:
    json.dump(AVAILABILITY, stream, indent=2)
print(json.dumps(AVAILABILITY, indent=2))
DEV_PATH = REPO_DIR / "data/splits/dev_items.csv"
TEST_PATH = REPO_DIR / "data/splits/test_items.csv"
def model_identity(system):
    return {key: QWEN_IDENTITY[key] for key in QWEN_STABLE_FIELDS} if system["name"] == "qwen" else None

## 5. Technical Development Pilot

Each system receives the same first ten development items in both tasks: **20 logical responses per system**. Candidate order is derived from the item identity and a fixed candidate-order seed of 42. Each request is journalled before execution and its outcome after return.

At most two attempts are allowed per logical request. Explicit provider blocks stop collection for that provider, and transient retries obey bounded waiting. Interrupted requests with uncertain completion are retained for inspection and are not automatically resent. Their possible expenditure remains included through conservative allowances.

Pilot and test journals for every system contribute to the shared expenditure ceiling, including systems not currently selected. Preserve all journals when resuming. An incomplete pilot does not qualify its system for test evaluation.


In [ ]:
# Upper input allowance: prompt UTF-8 bytes plus provider-envelope margin.
from hebrew_acronyms.models.common.pairs import load_rows
from hebrew_acronyms.models.common.eval import build_generate_prompt, build_select_prompt
all_rows = load_rows(DEV_PATH) + load_rows(TEST_PATH)
max_prompt_bytes = max(len(prompt.encode("utf-8")) for row in all_rows for prompt in (
    build_generate_prompt(row["acronym"], row["sentence"]),
    build_select_prompt(row["acronym"], row["sentence"], row["candidates"].split("|")),
))
input_token_allowance = max_prompt_bytes + 1024
if input_token_allowance >= 100_000:
    raise ValueError("Recheck long-context price tier before any calls")
OUTPUT_CAPS = {"qwen": 512, "gemini": 1024, "openai": 512, "anthropic": 1024}
RESERVE_PER_CALL_USD = {
    name: (input_token_allowance * rate["input"] + OUTPUT_CAPS[name] * rate["output"]) / 1_000_000
    for name, rate in RATES.items()
}
def current_session():
    return prepare_session(
        OUTPUT_ROOT, DEV_PATH, TEST_PATH, code_revision=CODE_REVISION,
        prior_spend_ils=PRIOR_SPEND_ILS, rates=RATES, reserves=RESERVE_PER_CALL_USD,
        ils_per_usd=ILS_PER_USD_ALLOWANCE,
    )
current_session()
for system in available_selected_systems():
    result = run_system_pilot(OUTPUT_ROOT, system, model_identity=model_identity(system))
    print(system["name"], result["manifest"]["identity_sha256"],
          {key: result["summary"][key] for key in ("n_calls", "n_completed", "n_pending", "n_ambiguous")})
print(json.dumps(session_summary(OUTPUT_ROOT), indent=2))

## 6. Pilot Inspection and Test Projection

Inspect all 20 responses for coverage, model identity, finish reason, truncation, token usage, elapsed time, and estimated cost. Pilot acceptance is technical: semantic errors remain observations and do not justify repeating requests or selecting a more favorable run.

For a complete pilot, the cost projection scales from 10 to 395 items with a 25% margin and includes accumulated expenditure across systems and allowances for uncertain usage. The elapsed-time projection is descriptive rather than a completion guarantee.

After inspecting a successful pilot's records, enter its printed `pilot_identity` in `INSPECTED_PILOTS` in the next cell. The initially empty mapping records no completed inspection. Incomplete or blocked systems remain reported, while independently qualified systems may proceed. Execute these stages sequentially; **Run all** is not a substitute for pilot inspection.


In [ ]:
current_session()  # Validate the saved session against the current configuration.
for system in available_selected_systems():
    name = system["name"]
    pilot_summary = summarize_evaluation(OUTPUT_ROOT / name / "dev-pilot")
    print(name, json.dumps(pilot_summary["by_system_task"], ensure_ascii=False))
    for record in pilot_summary["records"]:
        print(json.dumps(record, ensure_ascii=False))
    try:
        review = review_system_pilot(OUTPUT_ROOT, system, model_identity=model_identity(system))
        print(name, "pilot_identity:", review["pilot_identity"])
        print(json.dumps(review["cost"], indent=2))
        print("Projected full minutes:", round(pilot_summary["timings_seconds"][name] * 395 / 10 / 60, 1))
        print("Projected total ILS including all saved systems:", review["projected_total_ils"])
    except ValueError:
        print(name, "Full test gated: pilot incomplete or identity/configuration mismatch; inspect saved evidence.")
print(json.dumps(session_summary(OUTPUT_ROOT), indent=2))

## 7. Test Evaluation and Resumption

Each qualified system is evaluated on all **395 test items in both tasks**, yielding **790 logical response records**. `INSPECTED_PILOTS` maps each inspected system name to the exact SHA-256 identity printed for its successful pilot. Before collection, the runner reloads that pilot, validates source, inputs, settings, and model identity, checks coverage, and recomputes the projection and shared expenditure.

Each additional system requires its own matching pilot. Unselected, uninspected, or blocked systems are reported explicitly. Resumption retains completed responses, failed attempts, ambiguous requests, and accumulated expenditure; completed responses are not requested again. All test identities remain in the cohort, including failed and unrun cases.

Freeze the source revision and settings during collection. A configuration change or repair of an explicit account block requires a new identified session and matching pilots, with previous artifacts preserved and expenditure carried forward. Deleting journals or changing identifiers to bypass a block is incompatible with this protocol.


In [ ]:
INSPECTED_PILOTS = {}  # Map inspected system names to their exact pilot identity SHA-256.
current_session()  # Validate the saved session against the current configuration.
for system in available_selected_systems():
    name = system["name"]
    if name not in INSPECTED_PILOTS:
        print(name, "Full test not run: pilot not inspected")
        continue
    try:
        test_summary = run_system_full(
            OUTPUT_ROOT, system, inspected_pilot_identity=INSPECTED_PILOTS[name],
            model_identity=model_identity(system),
        )
        print(name, {key: test_summary[key] for key in
              ("n_items", "n_records", "n_calls", "n_completed", "n_pending", "n_ambiguous")})
    except ValueError:
        print(name, "Full test gated: verify saved pilot identity, coverage, inputs and shared budget.")
print(json.dumps(session_summary(OUTPUT_ROOT), indent=2))

## 8. Baselines and Encoder Limitations

The deterministic baseline evaluation includes uniform-random expectation, most-frequent expansion, most-mined expansion, and oracle selection. The trained DictaBERT candidate scorer and the untrained DictaBERT similarity baseline are separate systems; this notebook does not retrain either model.

The checkpoint adapter in `src/hebrew_acronyms/models/dictabert_cross_encoder/colab.py` supports the identified weights with SHA-256 `23bbff0324a7ce4d9d4a24a9ebfb87a4f6ac296bf1f1ce1bd25126c09260267f`. Checkpoint reuse also requires the corresponding model and tokenizer snapshot. A state dictionary alone does not establish the original training seed, selected epoch, library versions, or exact training examples. See `docs/checkpoints.md` in the source archive for the inference procedure and provenance limitations.

**Encoder evaluation limitation.** The historical test CSV lacks the explicit target spans required by the strict encoder interface. Multiple acronym occurrences cannot be resolved by silently selecting the first match. Fresh encoder evaluation requires qualified spans and is not performed by this notebook. Historical encoder results remain separately identified.


In [ ]:
from hebrew_acronyms.test_baselines import run_test_baselines
baseline_summary = run_test_baselines(
    TEST_PATH, REPO_DIR / "data/mined/candidate_table.csv",
    OUTPUT_ROOT / "baselines", code_revision=CODE_REVISION,
)
print(baseline_summary["summary"])

## 9. Result Artifacts and Interpretation

Each system's pilot and test directories contain a `manifest.json` identifying code, inputs, complete prompts, candidate orders, and settings; an `attempts.jsonl` journal retaining requests and outcomes; and a regenerated `summary.json` containing coverage and response records. `session.json` records the shared session configuration. Preserve the complete Drive directory and download an archive as a second copy.

Report results by system and task with the full cohort denominator, including missing or incomplete responses. Automatic selection-label parsing, string-based scores, and output-quality flags are distinct from human semantic judgments of generated expansions. Interpretation must retain source groups, returned model versions, run identities, and judgment provenance. Candidate labels beyond Z retain items with 30 candidates; comparisons with historical runs must account for this protocol difference.

`notebooks/experimental_study.ipynb` is the methods and analysis companion in the source archive. The existing development-set paper exporter is not a full-test exporter. Only explicitly identified run artifacts and evaluation procedures support reported results; downloading an archive does not itself validate the experiment or publish its findings.


In [ ]:
from google.colab import files
print(json.dumps(session_summary(OUTPUT_ROOT), ensure_ascii=False, indent=2))
for name in SYSTEM_NAMES:
    directory = OUTPUT_ROOT / name / "full-test"
    if (directory / "manifest.json").exists():
        saved_summary = summarize_evaluation(directory)
        print(json.dumps(saved_summary["by_system_task"], ensure_ascii=False, indent=2))
archive_path = shutil.make_archive("/content/" + RUN_NAME + "-" + str(time.time_ns()), "zip", OUTPUT_ROOT)
files.download(archive_path)